# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule**: We want to identify content that is visible to users but failing to capture clicks or engagement, as well as content that is almost ranking on page 1 but needs a push. A page is flagged if its search volume is > 100, and it either ranks top 10 with poor CTR (< 1%), or ranks in 'striking distance' (position 11-20).

**Reason Codes**:
- `top_10_low_ctr`: Ranks well but nobody is clicking (needs title/meta optimization).
- `striking_distance`: Position 11-20 with decent volume (needs content expansion or internal links).
The score is determined directly by `search_volume` so we prioritize higher-impact opportunities first.

In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Build the rule
def get_reason_code(row):
    if row['search_volume'] < 100 or row['avg_position'] == 0:
        return None
    if row['avg_position'] <= 10 and row['ctr'] < 1.0:
        return 'top_10_low_ctr'
    elif 10 < row['avg_position'] <= 20:
        return 'striking_distance'
    return None

df['reason_code'] = df.apply(get_reason_code, axis=1)
df['baseline_score'] = df['search_volume'].where(df['reason_code'].notnull(), 0)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


In [2]:
ranked_queue = df[df['baseline_score'] > 0].sort_values('baseline_score', ascending=False).copy()
ranked_queue['rank'] = range(1, len(ranked_queue) + 1)

os.makedirs('../outputs', exist_ok=True)
output_cols = ['rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'avg_position', 'ctr', 'search_volume', 'content_type']
ranked_queue[output_cols].to_csv('../outputs/baseline_action_score.csv', index=False)

print(f"Generated queue of {len(ranked_queue)} actionable items.")
print("Distribution of reason codes:")
print(ranked_queue['reason_code'].value_counts())

Generated queue of 1768 actionable items.
Distribution of reason codes:
reason_code
top_10_low_ctr       1003
striking_distance     765
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


In [3]:
top_20 = ranked_queue[output_cols].head(20)
print("Top 20 Review Sample (first 5):")
display(top_20.head(5))

# Review notes based on the data
print("\nReview Note for Rank 1:")
print(f"Content: {top_20.iloc[0]['content_id']}")
print(f"Action: Review for meta tag optimization or content expansion based on reason code '{top_20.iloc[0]['reason_code']}'.")
print("Confidence: Moderate. Based entirely on search volume, which may be noisy or not convert well.")
print("What would make it wrong: If the search intent is purely informational but the page is transactional, a better rank won't help.")

Top 20 Review Sample (first 5):


,rank,content_id,client_id,baseline_score,reason_code,avg_position,ctr,search_volume,content_type
13502,1,content_f76ccf7a7834,client_19581e27de,49500.0,top_10_low_ctr,9.5,0.15,49500.0,keyword article
5287,2,content_8ca50876b0df,client_3fdba35f04,40500.0,striking_distance,18.3,0.03,40500.0,keyword article
16585,3,content_19bdaa296a9b,client_19581e27de,33100.0,striking_distance,13.5,0.00,33100.0,keyword article
2553,4,content_eb1510f4b5f1,client_3fdba35f04,33100.0,striking_distance,14.7,0.00,33100.0,keyword article
10466,5,content_71f8734aebe2,client_8722616204,27100.0,striking_distance,15.0,0.00,27100.0,keyword article



Review Note for Rank 1:
Content: content_f76ccf7a7834
Action: Review for meta tag optimization or content expansion based on reason code 'top_10_low_ctr'.
Confidence: Moderate. Based entirely on search volume, which may be noisy or not convert well.
What would make it wrong: If the search intent is purely informational but the page is transactional, a better rank won't help.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Leakage Check**: We used only trailing 90-day metrics (`search_volume`, `avg_position`, `ctr`). No future windows or predefined categorical outcome labels were leaked.
**Weak Picks**: Ranking purely by `search_volume` can be dangerous. Some items might have massive search volume but are dominated by branded searches belonging to another company (e.g., "facebook login"), meaning our client will never realistically rank #1 or get clicks for it even if they are currently at position 12. Also, short-form content might legitimately have low CTR if the answer is featured directly on the search engine results page (SERP).

In [4]:
# Leakage check: Ensure 'is_declining_label' or 'trend_direction' was not used
used_columns = ['search_volume', 'avg_position', 'ctr']
print("Columns used for baseline score logic:", used_columns)

Columns used for baseline score logic: ['search_volume', 'avg_position', 'ctr']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.